In [40]:
import json
import logging
import os
import random
import sys
import warnings
from dataclasses import dataclass, field
from typing import Dict, List, Tuple

import joblib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import torch
import torch.nn as nn
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    precision_recall_curve,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, RobustScaler
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", stream=sys.stdout)
logger = logging.getLogger(__name__)

In [41]:
@dataclass
class AnomalyConfig:
    seed: int = 42
    dataset_path: str = "logging_monitoring_anomalies.csv"
    artifact_dir: str = "./anomaly_artifacts"
    n_anomalies_per_type: int = 500

    patterns: Dict = field(default_factory=lambda: {
        "resource_exhaustion": {
            "CPU_Usage_Percent": (98.0, 100.0), "Memory_Usage_MB": (62000, 63999),
            "Disk_Usage_Percent": (97.0, 100.0), "Response_Time_ms": (9700, 9999),
            "Escalation_Level": (4, 4), "Retry_Count": (9, 9), "Affected_Services": (18, 19),
        },
        "brute_force": {
            "Login_Attempts": (48, 49), "Failed_Transactions": (18, 19),
            "Alert_Count": (47, 49), "Response_Time_ms": (9500, 9999),
            "Retry_Count": (9, 9), "Escalation_Level": (4, 4), "Disk_Usage_Percent": (90.0, 100.0),
        },
        "data_exfiltration": {
            "Network_Out_KB": (980000, 999991), "Network_In_KB": (2, 500),
            "Anomaly_Duration_sec": (6900, 7199), "Login_Attempts": (47, 49),
            "Escalation_Level": (4, 4), "Alert_Count": (46, 49),
        },
        "network_flood": {
            "Network_In_KB": (985000, 999997), "Network_Out_KB": (985000, 999991),
            "Affected_Services": (18, 19), "Response_Time_ms": (9500, 9999),
            "Alert_Count": (47, 49), "Escalation_Level": (4, 4), "Retry_Count": (9, 9),
        },
        "disk_failure": {
            "Disk_Usage_Percent": (98.0, 100.0), "Anomaly_Duration_sec": (6800, 7199),
            "Retry_Count": (9, 9), "Failed_Transactions": (18, 19),
            "Response_Time_ms": (9500, 9999), "Escalation_Level": (4, 4), "CPU_Usage_Percent": (92.0, 100.0),
        },
        "security_breach": {
            "Login_Attempts": (48, 49), "Network_Out_KB": (970000, 999991),
            "Failed_Transactions": (18, 19), "Escalation_Level": (4, 4),
            "Alert_Count": (47, 49), "Retry_Count": (9, 9), "CPU_Usage_Percent": (90.0, 100.0),
        },
    })

    # Model Hyperparameters
    if_estimators: int = 300
    if_contamination: float = 0.03
    if_max_features: float = 0.8

    ae_hidden_dims: List[int] = field(default_factory=lambda: [64, 32])
    ae_latent_dim: int = 8
    ae_dropout: float = 0.05
    ae_epochs: int = 60
    ae_batch_size: int = 512
    ae_lr: float = 3e-4
    ae_weight_decay: float = 1e-5
    ae_patience: int = 15

    w_if: float = 0.40
    w_ae: float = 0.60
    use_pr_threshold: bool = True

config = AnomalyConfig()
os.makedirs(config.artifact_dir, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [42]:
def set_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

def min_max_scale(data: np.ndarray) -> np.ndarray:
    return (data - data.min()) / (data.max() - data.min() + 1e-12)

class FeatureProcessor:
    NUM_COLS = [
        'Response_Time_ms', 'Resolution_Time_min', 'Affected_Services',
        'CPU_Usage_Percent', 'Memory_Usage_MB', 'Disk_Usage_Percent',
        'Network_In_KB', 'Network_Out_KB', 'Login_Attempts',
        'Failed_Transactions', 'Anomaly_Duration_sec', 'Patch_Level',
        'Alert_Count', 'Retry_Count', 'Escalation_Level'
    ]
    CAT_COLS = [
        'Anomaly_Type', 'Status', 'Source', 'Alert_Method',
        'User_Role', 'TimeZone', 'Location', 'Service_Type'
    ]

    def __init__(self):
        self.encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        self.scaler = RobustScaler()
        self.feature_names = []

    def _generate_features(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        ts = pd.to_datetime(df['Timestamp'])
        df['hour'] = ts.dt.hour
        df['day_of_week'] = ts.dt.dayofweek
        df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
        df['net_io_ratio'] = np.log1p((df['Network_Out_KB'] + 1) / (df['Network_In_KB'] + 1))
        df['hw_pressure'] = (df['CPU_Usage_Percent'] + df['Disk_Usage_Percent']) / 2
        df['auth_fail_rate'] = (df['Login_Attempts'] + 1e-3) / (df['Alert_Count'] + 1)
        df['severity_idx'] = df['Retry_Count'] * df['Escalation_Level']

        for col in ['Response_Time_ms', 'Network_In_KB', 'Network_Out_KB', 'Memory_Usage_MB']:
            df[f'log_{col}'] = np.log1p(df[col])
        return df

    def fit_transform(self, df: pd.DataFrame) -> np.ndarray:
        df_proc = self._generate_features(df)
        cat_encoded = self.encoder.fit_transform(df_proc[self.CAT_COLS].astype(str))
        encoded_names = [f"{c}_enc" for c in self.CAT_COLS]
        for i, col in enumerate(encoded_names):
            df_proc[col] = cat_encoded[:, i]

        derived = ['hour', 'day_of_week', 'is_weekend', 'net_io_ratio', 'hw_pressure', 'auth_fail_rate', 'severity_idx']
        logs = [f'log_{c}' for c in ['Response_Time_ms', 'Network_In_KB', 'Network_Out_KB', 'Memory_Usage_MB']]
        self.feature_names = self.NUM_COLS + derived + logs + encoded_names
        return self.scaler.fit_transform(df_proc[self.feature_names].values).astype(np.float32)

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        df_proc = self._generate_features(df)
        cat_encoded = self.encoder.transform(df_proc[self.CAT_COLS].astype(str))
        for i, col in enumerate([f"{c}_enc" for c in self.CAT_COLS]):
            df_proc[col] = cat_encoded[:, i]
        return self.scaler.transform(df_proc[self.feature_names].values).astype(np.float32)

In [43]:
class Autoencoder(nn.Module):
    def __init__(self, in_dim: int, h_dims: List[int], z_dim: int, dropout: float):
        super().__init__()
        self.input_norm = nn.LayerNorm(in_dim)

        # Encoder
        enc_layers = []
        curr_dim = in_dim
        for h in h_dims:
            enc_layers.extend([nn.Linear(curr_dim, h), nn.LayerNorm(h), nn.GELU(), nn.Dropout(dropout)])
            curr_dim = h
        enc_layers.append(nn.Linear(curr_dim, z_dim))
        self.encoder = nn.Sequential(*enc_layers)

        # Decoder
        dec_layers = []
        curr_dim = z_dim
        for h in reversed(h_dims):
            dec_layers.extend([nn.Linear(curr_dim, h), nn.LayerNorm(h), nn.GELU(), nn.Dropout(dropout)])
            curr_dim = h
        dec_layers.append(nn.Linear(curr_dim, in_dim))
        self.decoder = nn.Sequential(*dec_layers)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        z = self.encoder(self.input_norm(x))
        return self.decoder(z), z

In [44]:
def train_ae(model, train_data, val_data, cfg, device):
    train_loader = DataLoader(TensorDataset(torch.from_numpy(train_data)), batch_size=cfg.ae_batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(torch.from_numpy(val_data)), batch_size=cfg.ae_batch_size)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.ae_lr, weight_decay=cfg.ae_weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.ae_epochs)
    crit = nn.MSELoss()

    best_loss, patience_counter = float('inf'), 0
    path = "/tmp/ae_best.pt"

    for epoch in range(cfg.ae_epochs):
        model.train()
        for (bx,) in train_loader:
            bx = bx.to(device); opt.zero_grad()
            rec, _ = model(bx); loss = crit(rec, bx)
            loss.backward(); opt.step()

        sched.step(); model.eval(); val_l = 0
        with torch.no_grad():
            for (bx,) in val_loader:
                bx = bx.to(device); rec, _ = model(bx)
                val_l += crit(rec, bx).item()

        avg_l = val_l / len(val_loader)
        if avg_l < best_loss:
            best_loss, patience_counter = avg_l, 0
            torch.save(model.state_dict(), path)
        else:
            patience_counter += 1
        if patience_counter >= cfg.ae_patience: break

    model.load_state_dict(torch.load(path)); return model

@torch.no_grad()
def get_ae_scores(model, data, device):
    model.eval()
    x = torch.from_numpy(data).to(device)
    rec, _ = model(x)
    mse = torch.mean((x - rec)**2, dim=1).cpu().numpy()
    feat_mse = ((x - rec)**2).cpu().numpy()
    return mse, feat_mse

In [45]:
set_seeds(config.seed)

# Load and inject synthetic anomalies
df_raw = pd.read_csv(config.dataset_path)
df_raw['target'], df_raw['type'] = 0, 'normal'

for a_type, ranges in config.patterns.items():
    idx = np.random.choice(df_raw.index, config.n_anomalies_per_type, replace=False)
    for feat, (low, high) in ranges.items():
        df_raw.loc[idx, feat] = np.random.uniform(low, high, len(idx))
    df_raw.loc[idx, 'target'], df_raw.loc[idx, 'type'] = 1, a_type

# Preprocessing
df_train, df_test = train_test_split(df_raw, test_size=0.2, stratify=df_raw['target'], random_state=config.seed)
proc = FeatureProcessor()
x_train, x_test = proc.fit_transform(df_train), proc.transform(df_test)

# Isolation Forest
if_m = IsolationForest(n_estimators=config.if_estimators, contamination=config.if_contamination, random_state=config.seed, n_jobs=-1)
if_m.fit(x_train[df_train['target'] == 0])
if_test_s = min_max_scale(-if_m.score_samples(x_test))
if_train_s = min_max_scale(-if_m.score_samples(x_train))

# Autoencoder
ae_m = Autoencoder(x_train.shape[1], config.ae_hidden_dims, config.ae_latent_dim, config.ae_dropout).to(device)
ae_m = train_ae(ae_m, x_train[df_train['target'] == 0], x_train[df_train['target'] == 0][:1000], config, device)
ae_test_s, _ = get_ae_scores(ae_m, x_test, device)
ae_train_s, _ = get_ae_scores(ae_m, x_train, device)
ae_test_s, ae_train_s = min_max_scale(ae_test_s), min_max_scale(ae_train_s)

In [46]:
# Calculate ensemble scores for test and train sets
ens_test = (config.w_if * if_test_s) + (config.w_ae * ae_test_s)
ens_train = (config.w_if * if_train_s) + (config.w_ae * ae_train_s)

# Threshold optimization
if config.use_pr_threshold:
    prec_tr, rec_tr, thresh_tr = precision_recall_curve(df_train['target'], ens_train)
    f1_tr = 2 * (prec_tr * rec_tr) / (prec_tr + rec_tr + 1e-12)
    best_idx = np.argmax(f1_tr[:-1])
    thresh = thresh_tr[best_idx]
    logger.info(f"Optimal Threshold (F1-Maximized): {thresh:.4f}")
else:
    thresh = np.percentile(ens_train, 97)
    logger.info(f"Percentile-Based Threshold: {thresh:.4f}")

# Generate binary predictions
y_pred = (ens_test >= thresh).astype(int)

# Evaluation metrics
metrics = {
    "ROC_AUC": roc_auc_score(df_test['target'], ens_test),
    "F1_Score": f1_score(df_test['target'], y_pred),
    "Precision": precision_score(df_test['target'], y_pred, zero_division=0),
    "Recall": recall_score(df_test['target'], y_pred)
}

# Print metrics to console
print("-" * 30)
print("FINAL PIPELINE METRICS")
print("-" * 30)
for k, v in metrics.items():
    print(f"{k:15}: {v:.4f}")
print("-" * 30)

# --- Save Artifacts to Disk ---
# 1. Save Feature Processor (includes scaler and encoder)
joblib.dump(proc, os.path.join(config.artifact_dir, "processor.joblib"))

# 2. Save Isolation Forest Model
joblib.dump(if_m, os.path.join(config.artifact_dir, "if_model.joblib"))

# 3. Save Autoencoder State Dict (Weights)
torch.save(ae_m.state_dict(), os.path.join(config.artifact_dir, "ae_weights.pt"))

# 4. Save Metrics as JSON
with open(os.path.join(config.artifact_dir, "metrics.json"), 'w') as f:
    json.dump({k: float(v) for k, v in metrics.items()}, f, indent=4)

logger.info(f"All artifacts saved successfully to {config.artifact_dir}")

------------------------------
FINAL PIPELINE METRICS
------------------------------
ROC_AUC        : 0.9609
F1_Score       : 0.4807
Precision      : 0.3805
Recall         : 0.6526
------------------------------


In [47]:
def plot_timeline(df, bucket='12h'):
    # Create a local copy and ensure Timestamp is a DatetimeIndex
    df_plot = df.copy()
    df_plot['Timestamp'] = pd.to_datetime(df_plot['Timestamp'])
    df_plot = df_plot.set_index('Timestamp')

    # Resample to the specified time bucket
    resampled = df_plot.resample(bucket).agg({
        'pred_flag': 'mean',
        'target': 'mean',
        'ensemble_score': 'mean'
    }).dropna()

    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        subplot_titles=["Anomaly Rate (%)", "Mean Ensemble Score"]
    )

    # Trace 1: Actual vs Predicted rates
    fig.add_trace(go.Scatter(
        x=resampled.index, y=resampled['target']*100,
        name="Actual", fill="tozeroy", line=dict(color='#e74c3c')
    ), 1, 1)

    fig.add_trace(go.Scatter(
        x=resampled.index, y=resampled['pred_flag']*100,
        name="Predicted", line=dict(dash="dash", color='#f39c12')
    ), 1, 1)

    # Trace 2: Ensemble Score
    fig.add_trace(go.Scatter(
        x=resampled.index, y=resampled['ensemble_score'],
        name="Score", fill="tozeroy", line=dict(color='#9b59b6')
    ), 2, 1)

    fig.update_layout(template="plotly_dark", height=500, title="<b>Rolling Anomaly Monitoring</b>")
    return fig

# Prepare the results dataframe for the plots
df_results = df_test.copy()
df_results['ensemble_score'] = ens_test
df_results['ae_error'] = ae_test_s
df_results['pred_flag'] = y_pred

plot_timeline(df_results).show()

In [48]:
import plotly.express as px

def plot_landscape(df, sample_frac=0.05):
    # Keep all predicted anomalies, sample normal points for performance
    df_anomalies = df[df['pred_flag'] == 1]
    df_normal = df[df['pred_flag'] == 0].sample(frac=sample_frac, random_state=42)
    df_render = pd.concat([df_anomalies, df_normal])

    fig = px.scatter(
        df_render,
        x="ensemble_score",
        y="ae_error",
        color="type",
        title="<b>Anomaly Landscape: Ensemble Score vs AE Error</b>",
        template="plotly_dark",
        height=500,
        opacity=0.7
    )
    return fig

plot_landscape(df_results).show()

In [49]:
def plot_root_cause(df, feat_mse, feat_names, top_k=100):
    # Sort by score to find the most significant anomalies
    top_indices = np.argsort(df['ensemble_score'].values)[::-1][:top_k]

    # Calculate mean reconstruction error per feature
    mean_errors = feat_mse[top_indices].mean(axis=0)
    sort_idx = np.argsort(mean_errors)[::-1][:20]

    fig = go.Figure(go.Bar(
        x=mean_errors[sort_idx],
        y=[feat_names[i] for i in sort_idx],
        orientation='h',
        marker=dict(color=mean_errors[sort_idx], colorscale='YlOrRd', showscale=True)
    ))

    fig.update_layout(
        title=f"<b>Root Cause: Feature Contribution (Top {top_k} Anomalies)</b>",
        xaxis_title="Mean Reconstruction Error (MSE)",
        template="plotly_dark",
        height=520,
        margin=dict(l=200)
    )
    return fig

# Calculate feature-level MSE for the test set
_, feature_mse_test = get_ae_scores(ae_m, x_test, device)
plot_root_cause(df_results, feature_mse_test, proc.feature_names).show()

In [50]:
def load_inference_pipeline(artifact_dir, in_dim, cfg, device):
    """Loads all saved artifacts and initializes models for inference."""
    # Load feature processor
    proc = joblib.load(os.path.join(artifact_dir, "processor.joblib"))

    # Load Isolation Forest
    if_m = joblib.load(os.path.join(artifact_dir, "if_model.joblib"))

    # Initialize and load Autoencoder weights
    ae_m = Autoencoder(in_dim, cfg.ae_hidden_dims, cfg.ae_latent_dim, cfg.ae_dropout).to(device)
    ae_m.load_state_dict(torch.load(os.path.join(artifact_dir, "ae_weights.pt"), map_location=device))
    ae_m.eval()

    return proc, if_m, ae_m

def predict_anomaly(log_data: dict, proc, if_m, ae_m, cfg, threshold, device):
    """Predicts if a single log entry is an anomaly."""
    # Convert dict to DataFrame for the processor
    df_single = pd.DataFrame([log_data])

    # Process features
    x_scaled = proc.transform(df_single)

    # Get Isolation Forest score
    if_score = -if_m.score_samples(x_scaled)[0]

    # Get Autoencoder score
    with torch.no_grad():
        x_tensor = torch.from_numpy(x_scaled).to(device)
        recon, _ = ae_m(x_tensor)
        ae_mse = torch.mean((x_tensor - recon)**2).item()

    # Normalize scores (Note: Real production requires stored min/max values from training)
    # For this example, we assume scores are already in the expected range
    ensemble_score = (cfg.w_if * if_score) + (cfg.w_ae * ae_mse)

    return {
        "is_anomaly": bool(ensemble_score >= threshold),
        "score": float(ensemble_score),
        "if_contribution": float(if_score),
        "ae_contribution": float(ae_mse)
    }

# --- Example Usage ---
# 1. Setup paths and params
artifact_dir = config.artifact_dir
input_dim = x_train.shape[1]

# 2. Load models
inf_proc, inf_if, inf_ae = load_inference_pipeline(artifact_dir, input_dim, config, device)

# 3. Simulate a new incoming log
sample_log = df_test.iloc[0].to_dict()

# 4. Run inference
result = predict_anomaly(sample_log, inf_proc, inf_if, inf_ae, config, thresh, device)

print("-" * 30)
print("INFERENCE RESULT")
print("-" * 30)
for k, v in result.items():
    print(f"{k:15}: {v}")

------------------------------
INFERENCE RESULT
------------------------------
is_anomaly     : True
score          : 0.3057134143076851
if_contribution: 0.5299525910522824
ae_contribution: 0.15622062981128693
